In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

In [2]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
CUDA available: False


In [4]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [5]:
from google.colab import files

uploaded = files.upload()

Saving english_singlish_dataset_with_sentiment.csv to english_singlish_dataset_with_sentiment.csv


In [6]:
df = pd.read_csv("english_singlish_dataset_with_sentiment.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset shape: (126, 4)

Columns:
['English', 'Singlish_Word_Phrase', 'Context_Usage', 'Sentiment Analysis']


In [7]:
# Keep only the translation columns
data = df[["English", "Singlish_Word_Phrase"]].copy()

In [8]:
# Rename columns for clarity
data = data.rename(columns={
    "English": "source_text",
    "Singlish_Word_Phrase": "target_text"
})

In [9]:
# Remove missing values
data = data.dropna(subset=["source_text", "target_text"])

In [10]:
# Convert to string
data["source_text"] = data["source_text"].astype(str).str.strip()
data["target_text"] = data["target_text"].astype(str).str.strip()

In [11]:
# Remove empty records
data = data[
    (data["source_text"] != "") &
    (data["target_text"] != "")
]

In [12]:
# Remove exact duplicate translation pairs
data = data.drop_duplicates(
    subset=["source_text", "target_text"]
).reset_index(drop=True)

In [13]:
print("Clean dataset size:", len(data))
display(data.head(10))

Clean dataset size: 126


,source_text,target_text
0,Welcome / Hello,Ayubowan
1,Thank you,Istuti
2,How are you?,Kohomada?
3,I am fine.,Mama hodinn innawa.
4,What is your name?,Oyaage nama mokakda?
5,My name is...,Mage nama...
6,Where are you going?,Oya koheda yanne?
7,I am going home.,Mama gedara yanawa.
8,How much is this?,Meka kiyada?
9,It is too expensive.,Meka gana wadi.


In [14]:
# print(data.isnull().sum())
# print("Duplicates:", data.duplicated().sum())

In [15]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, temp_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 100
Validation: 13
Testing: 13


In [16]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['source_text', 'target_text'],
    num_rows: 100
})
Dataset({
    features: ['source_text', 'target_text'],
    num_rows: 13
})
Dataset({
    features: ['source_text', 'target_text'],
    num_rows: 13
})


In [17]:
MODEL_NAME = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Model loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/mt5-small


In [18]:
MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64

def preprocess_function(examples):
    inputs = [
        f"translate English to Singlish: {text}"
        for text in examples["source_text"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [19]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

print(tokenized_train)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})


In [20]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [21]:
OUTPUT_DIR = "./mt5-english-singlish"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=10,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=5e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    predict_with_generate=True,

    logging_strategy="steps",
    logging_steps=10,

    save_total_limit=2,

    report_to="none",

    fp16=torch.cuda.is_available(),
)

In [22]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,
    data_collator=data_collator,
)

In [23]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.16.1


In [24]:
import inspect
print(inspect.signature(Seq2SeqTrainer.__init__))

(self, model: Union[ForwardRef('PreTrainedModel'), torch.nn.modules.module.Module, NoneType] = None, args: Optional[ForwardRef('TrainingArguments')] = None, data_collator: Optional[ForwardRef('DataCollator')] = None, train_dataset: Union[torch.utils.data.dataset.Dataset, ForwardRef('IterableDataset'), ForwardRef('datasets.Dataset'), NoneType] = None, eval_dataset: torch.utils.data.dataset.Dataset | dict[str, torch.utils.data.dataset.Dataset] | None = None, processing_class: Union[ForwardRef('PreTrainedTokenizerBase'), ForwardRef('BaseImageProcessor'), ForwardRef('FeatureExtractionMixin'), ForwardRef('ProcessorMixin'), NoneType] = None, model_init: collections.abc.Callable[[], 'PreTrainedModel'] | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[['EvalPrediction'], dict] | None = None, callbacks: list['TrainerCallback'] | None = None, optimizers: tuple[torch.optim.optimizer.Optimizer | None, torch.optim.lr_scheduler.Lambda

In [25]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,25.752075,20.047379
2,23.438533,19.657764
3,22.916182,17.127262
4,19.923865,15.581446
5,19.599908,13.798475
6,18.901024,14.280821
7,18.278473,13.720798
8,17.272221,13.899729
9,17.659544,13.626556
10,18.562285,13.598827


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=250, training_loss=20.852809997558595, metrics={'train_runtime': 3044.689, 'train_samples_per_second': 0.328, 'train_steps_per_second': 0.082, 'total_flos': 25132133744640.0, 'train_loss': 20.852809997558595, 'epoch': 10.0})

In [26]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_test
)

print(test_results)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
18.562285,12.497030,10


{'eval_loss': 12.497030258178711}


In [27]:
def translate_english_to_singlish(text):
    prompt = f"translate English to Singlish: {text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4
        )

    translation = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return translation

In [28]:
examples = [
    "How are you?",
    "Thank you",
    "Where are you going?",
    "I am fine.",
    "What is your name?"
]

for text in examples:
    print("English :", text)
    print("Singlish:", translate_english_to_singlish(text))
    print("-" * 50)

English : How are you?
Singlish: <extra_id_0>
--------------------------------------------------
English : Thank you
Singlish: <extra_id_0>
--------------------------------------------------
English : Where are you going?
Singlish: <extra_id_0>.
--------------------------------------------------
English : I am fine.
Singlish: <extra_id_0>
--------------------------------------------------
English : What is your name?
Singlish: <extra_id_0>?
--------------------------------------------------


In [29]:
SAVE_PATH = "./english-singlish-mt5"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./english-singlish-mt5


In [30]:
from google.colab import files
import shutil
import os

# Create a final project folder
final_folder = "/content/final_training_output"
os.makedirs(final_folder, exist_ok=True)

# Save final dataset
data.to_csv(
    f"{final_folder}/final_english_singlish_translation_dataset.csv",
    index=False
)

# Save trained model
trainer.save_model(f"{final_folder}/english-singlish-mt5")
tokenizer.save_pretrained(f"{final_folder}/english-singlish-mt5")

# Compress everything
zip_path = shutil.make_archive(
    "/content/english-singlish-training-final",
    "zip",
    final_folder
)

# Download ZIP
files.download(zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 3.9 MB/s eta 0:00:00


In [32]:
from huggingface_hub import login

login()

In [33]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "Madushanavod/english-singlish-mt5"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

api.upload_folder(
    folder_path="/content/english-singlish-mt5",
    repo_id=repo_id,
    repo_type="model"
)

print("Model uploaded successfully!")

Model uploaded successfully!
